# Анализ теста на реальных данных

Визуализация результатов `07_real_data_test.py`.

Сопоставление номенклатуры предприятия (40K строк) со сводной спецификацией (18K строк) — без разметки.

Метрики:
- **Category consistency** — совпадение категории спецификации с наименованием номенклатуры
- **Reciprocal matching** — доля взаимных nearest-neighbor пар
- **Confidence separation** — разрыв sim(top-1) vs sim(top-2)
- **GNN vs Baseline** — сравнение подходов

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["figure.figsize"] = (12, 5)

OUTPUT_DIR = Path("../output/07_real_data_test")

with open(OUTPUT_DIR / "results.json", encoding="utf-8") as f:
    results = json.load(f)

methods = [m for m in ("GNN", "Baseline") if m in results]
print(f"Методы: {methods}")

## 1. Сводная таблица метрик: GNN vs Baseline

In [ ]:
rows = []
for m in methods:
    r = results[m]
    rows.append({
        "Метод": m,
        "Category consistency": r["category_consistency"].get("precision", None),
        "Reciprocal rate": r["reciprocal"]["rate"],
        "Mean confidence gap": r["confidence"]["mean_gap"],
        "Mean top-1 similarity": r["confidence"]["mean_top1_sim"],
        "% gap > 0.1": r["confidence"]["pct_gap_gt_01"],
        "% gap > 0.2": r["confidence"]["pct_gap_gt_02"],
    })

df_summary = pd.DataFrame(rows).set_index("Метод")
display(df_summary.style
    .format({
        "Category consistency": "{:.1%}",
        "Reciprocal rate": "{:.1%}",
        "Mean confidence gap": "{:.4f}",
        "Mean top-1 similarity": "{:.4f}",
        "% gap > 0.1": "{:.1%}",
        "% gap > 0.2": "{:.1%}",
    })
    .set_caption("Сводка: GNN vs Baseline (raw CLS cosine similarity)")
)

## 2. Category consistency по категориям

In [ ]:
for method in methods:
    per_cat = results[method]["category_consistency"].get("per_category", {})
    if not per_cat:
        continue
    
    df_cat = pd.DataFrame([
        {"Категория": k, "Total": v["total"], "Match": v["match"], "Precision": v.get("precision", 0)}
        for k, v in per_cat.items()
    ]).sort_values("Total", ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(df_cat) * 0.3)))
    
    # Bar chart: precision per category
    ax = axes[0]
    colors = ["C2" if p >= 0.5 else "C3" if p < 0.2 else "C1" for p in df_cat["Precision"]]
    ax.barh(df_cat["Категория"], df_cat["Precision"], color=colors, alpha=0.8, edgecolor="white")
    for i, (_, row) in enumerate(df_cat.iterrows()):
        ax.text(row["Precision"] + 0.01, i, f"{row['Precision']:.0%} ({row['Match']}/{row['Total']})",
                va="center", fontsize=8)
    ax.set_xlabel("Precision")
    ax.set_xlim(0, 1.3)
    ax.set_title(f"Category consistency — {method}")
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis="x")
    
    # Bar chart: count per category
    ax = axes[1]
    ax.barh(df_cat["Категория"], df_cat["Total"], color="C0", alpha=0.6, edgecolor="white")
    ax.set_xlabel("Количество строк")
    ax.set_title("Распределение по категориям")
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, axis="x")
    
    plt.tight_layout()
    plt.show()
    
    overall = results[method]["category_consistency"]["precision"]
    print(f"{method} — Overall category consistency: {overall:.1%}")
    print()

## 3. Confidence gap: гистограммы

In [ ]:
# Загружаем сохранённые гистограммы
fig, axes = plt.subplots(1, len(methods), figsize=(7 * len(methods), 4))
if len(methods) == 1:
    axes = [axes]

for ax, method in zip(axes, methods):
    img_path = OUTPUT_DIR / f"confidence_gap_{method}.png"
    if img_path.exists():
        img = plt.imread(str(img_path))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(method)
    else:
        ax.text(0.5, 0.5, f"Нет {img_path.name}", ha="center", va="center")
        ax.set_title(method)

plt.tight_layout()
plt.show()

## 4. Сравнение GNN vs Baseline (bar chart)

In [ ]:
if len(methods) == 2:
    metrics_names = ["Category\nconsistency", "Reciprocal\nrate", "Mean\nconfidence gap", "Mean\ntop-1 sim"]
    gnn_vals = [
        results["GNN"]["category_consistency"].get("precision", 0),
        results["GNN"]["reciprocal"]["rate"],
        results["GNN"]["confidence"]["mean_gap"],
        results["GNN"]["confidence"]["mean_top1_sim"],
    ]
    base_vals = [
        results["Baseline"]["category_consistency"].get("precision", 0),
        results["Baseline"]["reciprocal"]["rate"],
        results["Baseline"]["confidence"]["mean_gap"],
        results["Baseline"]["confidence"]["mean_top1_sim"],
    ]

    x = np.arange(len(metrics_names))
    w = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - w/2, gnn_vals, w, label="GNN", color="C0", alpha=0.8)
    bars2 = ax.bar(x + w/2, base_vals, w, label="Baseline", color="C3", alpha=0.8)

    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                    f"{h:.3f}", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics_names)
    ax.set_title("GNN vs Baseline — все метрики")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()

    # Разница
    print("Разница (GNN - Baseline):")
    for name, g, b in zip(metrics_names, gnn_vals, base_vals):
        diff = g - b
        sign = "+" if diff > 0 else ""
        print(f"  {name.replace(chr(10), ' '):25s}: {sign}{diff:.4f}")
else:
    print("Только один метод — сравнение недоступно")

## 5. Примеры матчей (sample_matches.csv)

In [ ]:
matches_path = OUTPUT_DIR / "sample_matches.csv"
if matches_path.exists():
    df_matches = pd.read_csv(matches_path)
    print(f"Всего строк: {len(df_matches)}")
    print(f"Уникальных spec строк: {df_matches['spec_idx'].nunique()}")
    print(f"Методы: {df_matches['method'].unique().tolist()}")
    print()
    
    # Показать первые 10 примеров GNN top-1
    gnn_top1 = df_matches[(df_matches["method"] == "GNN") & (df_matches["rank"] == 1)].head(15)
    display(gnn_top1[["spec_наименование", "spec_Категория", "nom_Наименование", "nom_Артикул", "similarity"]].style
        .set_caption("GNN: top-1 матчи (первые 15)")
    )
else:
    print("sample_matches.csv не найден")

### Сравнение GNN vs Baseline на одних и тех же строках

In [ ]:
if matches_path.exists() and len(methods) == 2:
    # Для первых 10 spec строк показать top-1 обоих методов рядом
    sample_ids = df_matches["spec_idx"].unique()[:10]
    
    comparison_rows = []
    for sid in sample_ids:
        subset = df_matches[(df_matches["spec_idx"] == sid) & (df_matches["rank"] == 1)]
        spec_name = subset.iloc[0]["spec_наименование"] if len(subset) > 0 else ""
        spec_cat = subset.iloc[0]["spec_Категория"] if len(subset) > 0 else ""
        
        gnn_row = subset[subset["method"] == "GNN"]
        base_row = subset[subset["method"] == "Baseline"]
        
        comparison_rows.append({
            "Спецификация": spec_name[:60],
            "Категория": spec_cat,
            "GNN → Номенклатура": gnn_row.iloc[0]["nom_Наименование"][:60] if len(gnn_row) > 0 else "—",
            "GNN sim": gnn_row.iloc[0]["similarity"] if len(gnn_row) > 0 else "—",
            "Base → Номенклатура": base_row.iloc[0]["nom_Наименование"][:60] if len(base_row) > 0 else "—",
            "Base sim": base_row.iloc[0]["similarity"] if len(base_row) > 0 else "—",
        })
    
    df_cmp = pd.DataFrame(comparison_rows)
    display(df_cmp.style.set_caption("Top-1 матчи: GNN vs Baseline (первые 10 строк)"))
else:
    print("Данные для сравнения недоступны")

## 6. Распределение similarity scores

In [ ]:
if matches_path.exists():
    fig, axes = plt.subplots(1, len(methods), figsize=(7 * len(methods), 4))
    if len(methods) == 1:
        axes = [axes]
    
    for ax, method in zip(axes, methods):
        top1 = df_matches[(df_matches["method"] == method) & (df_matches["rank"] == 1)]
        sims = top1["similarity"].astype(float)
        
        ax.hist(sims, bins=50, alpha=0.7, edgecolor="black", color="C0" if method == "GNN" else "C3")
        ax.axvline(sims.median(), color="red", ls="--", label=f"median={sims.median():.3f}")
        ax.axvline(sims.mean(), color="orange", ls="--", label=f"mean={sims.mean():.3f}")
        ax.set_xlabel("Cosine similarity (top-1)")
        ax.set_ylabel("Количество")
        ax.set_title(f"Распределение top-1 similarity — {method}")
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Итоговая сводка

In [ ]:
print("=" * 60)
print("ТЕСТ НА РЕАЛЬНЫХ ДАННЫХ — СВОДКА")
print("=" * 60)

for method in methods:
    r = results[method]
    cat_p = r["category_consistency"].get("precision", 0)
    rec_r = r["reciprocal"]["rate"]
    conf_g = r["confidence"]["mean_gap"]
    sim_m = r["confidence"]["mean_top1_sim"]
    
    print(f"\n{method}:")
    print(f"  Category consistency: {cat_p:.1%}")
    print(f"  Reciprocal rate:      {rec_r:.1%}")
    print(f"  Mean confidence gap:  {conf_g:.4f}")
    print(f"  Mean top-1 similarity:{sim_m:.4f}")

if len(methods) == 2:
    print("\n" + "-" * 40)
    g = results["GNN"]
    b = results["Baseline"]
    cat_diff = g["category_consistency"].get("precision", 0) - b["category_consistency"].get("precision", 0)
    rec_diff = g["reciprocal"]["rate"] - b["reciprocal"]["rate"]
    print(f"GNN advantage:")
    print(f"  Category consistency: {'+' if cat_diff >= 0 else ''}{cat_diff:.1%}")
    print(f"  Reciprocal rate:      {'+' if rec_diff >= 0 else ''}{rec_diff:.1%}")

print("=" * 60)